# Local multi-turn SQL lab

This notebook is the shareable lab attached to the post's codebase. It runs a small in-memory SQLite experiment that compares candidate fine-tuning targets for multi-turn SQL analysis.

The lab uses `auto` runtime by default. It selects CUDA, MPS, or XPU when PyTorch detects an available accelerator, and falls back to CPU.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "notebooks").exists():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").exists():
            repo_root = candidate
            break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
from notebooks.blog_support import (
    accuracy_scorecard,
    claim_table,
    data_engineering_gates,
    endpoint_run_scorecard,
    lab_failure_trace,
    lab_method_scorecard,
    metric_dsl_demo,
    metric_dsl_eval_contract,
    planner_scorecard,
    prompt_optimization_findings,
    target_comparison,
    target_evidence_matrix,
)
from notebooks.labs.local_multiturn_sql_lab_support import run_multiturn_lab


## 1. Research question

Can a small specialized model learn the behavior and semantic concepts needed to compete with hosted systems on multi-turn data analysis?

In [ ]:
report = run_multiturn_lab(device_preference="auto")

print(f"Runtime used by the lab: {report['device'].label}")
print(f"Accelerator availability: {report['detected_accelerator'].label}")
print("CUDA/MPS/XPU status:")
for status in report["accelerator_report"]:
    print(f"- {status['label']} ({status['usage']})")
print(f"Scenario hash: {report['scenario_contract']['shared_input_sha256']}")

assert report["device"].kind in {"cpu", "cuda", "mps", "xpu"}
assert report["runtime_policy"]["accelerator_usage"] == "selected_if_available"


## 2. Why single-turn SQL fails here

A single complete question can often be answered with one SQL query. A conversational analysis has to carry state across turns: the metric, filter, grain, value mapping, and recovery state can all change independently.

In [ ]:
import pandas as pd

pd.DataFrame(report["walkthrough_sections"])[
    ["section_id", "reader_question", "takeaway", "next_artifact"]
]


## 3. Candidate fine-tuning targets

In [ ]:
pd.DataFrame(report["method_matrix"])


## 4. Target scorecard

The toy lab is not a benchmark. It makes each fine-tuning target inspectable, then connects the behavior to the larger repo evidence and next gate.

In [ ]:
pd.DataFrame([
    {"system": system, **metrics}
    for system, metrics in report["systems"].items()
])


In [ ]:
lab_method_scorecard()


In [ ]:
target_comparison()


In [ ]:
target_evidence_matrix()


## 5. Execution trace

The trace separates value correctness from context carryover, value grounding, metric preservation, and recovery after an empty-result turn.

In [ ]:
trace_columns = [
    "turn_id",
    "question",
    "system",
    "value_match",
    "context_carryover",
    "value_grounded",
    "measure_preserved",
    "recovery_success",
    "failure_type",
    "intermediate_plan",
    "sql",
]

pd.DataFrame(report["rows"])[trace_columns]


In [ ]:
lab_failure_trace()


## 6. Endpoint and planner evidence

These tables are the checked-in evidence behind the larger claims. Non-oracle rows are production-style proxy runs; oracle rows are diagnostic ceilings.

In [ ]:
endpoint_run_scorecard()


In [ ]:
accuracy_scorecard()


In [ ]:
planner_scorecard()


In [ ]:
claim_table()


## 7. Metric DSL checkpoint

A raw SQL target can return rows while erasing governed metric intent. The DSL target keeps `MEASURE(revenue)` until the semantic model compiles it.

In [ ]:
metric_dsl_demo()["compiled_sql"]


In [ ]:
metric_dsl_eval_contract()


## 8. DSPy boundary

Prompt search is useful as a harness, but the current evidence says the next useful DSPy target is the planner program, not more wording search for final SQL.

In [ ]:
prompt_optimization_findings()


## 9. Data engineering gates

These are the artifacts needed before the lab's method comparison can support a broader multi-turn SQL claim.

In [ ]:
data_engineering_gates()
